# 🚬 Smoking Detection Pipeline — Kaggle GPU
### **Mô hình chính:** YOLOv8s-P2 (Stride 4 head — Tối ưu cho vật thể nhỏ)  
### **Mô hình đối chứng:** YOLOv8s Baseline (Standard P3–P5)  
---
## ⚙️ CÀI ĐẶT TRÊN KAGGLE (BẮT BUỘC TRƯỚC KHI CHẠY):
Ở cột bên phải màn hình (**Notebook settings**):
1. **Accelerator:** Chọn **GPU T4 x2** (hoặc **GPU P100**)
2. **Internet:** Gạt sang **ON** (để cài đặt thư viện và tải pretrained backbone)
3. **Persistence:** Chọn **Files only** (để giữ nguyên các file checkpoint khi mất mạng)

## 📦 Cell 1: Cài đặt Thư viện & Kiểm tra GPU

In [4]:
!pip install -q ultralytics

import warnings
warnings.filterwarnings('ignore')
import torch

print('PyTorch Version:', torch.__version__)
print('CUDA Available: ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('✅ GPU Name:', torch.cuda.get_device_name(0))
else:
    print('⚠️ Chưa bật GPU! Hãy vào mục Notebook settings ở cột bên phải -> chọn Accelerator: GPU T4 x2')

PyTorch Version: 2.10.0+cu128
CUDA Available:  True
✅ GPU Name: Tesla T4


## 📂 Cell 3: Tự động Định vị Dataset & Chuẩn hóa Đường dẫn Kaggle
Cell này tự động quét trong `/kaggle/input/` để tìm file `data.yaml` từ dataset bạn đã upload, sau đó tự động sửa đường dẫn cho khớp 100% với hệ điều hành Linux của Kaggle (không bao giờ bị lỗi `FileNotFoundError`).

In [5]:
from pathlib import Path
import yaml

# 1. Tìm data.yaml trong /kaggle/input (loại trừ sanity_check_data)
data_yamls = [p for p in Path('/kaggle/input').rglob('data.yaml') if 'sanity' not in str(p)]

if not data_yamls:
    print('⚠️ Chưa tìm thấy dataset trong /kaggle/input!')
    print('👉 Hướng dẫn: Ở cột bên phải Kaggle, nhấn "+ Add Input" -> "Upload Dataset" -> tải file dataset.zip lên.')
else:
    orig_yaml = data_yamls[0]
    dataset_root = orig_yaml.parent
    print(f'✅ Đã tìm thấy dataset tại: {dataset_root}')
    
    # Đọc cấu hình gốc
    with open(orig_yaml, 'r') as f:
        cfg = yaml.safe_load(f)
    
    # Cập nhật trường 'path' thành đường dẫn thực tế trên Kaggle
    cfg['path'] = str(dataset_root)
    
    # Ghi file cấu hình chuẩn vào /kaggle/working/
    kaggle_yaml_path = Path('/kaggle/working/data_kaggle.yaml')
    with open(kaggle_yaml_path, 'w') as f:
        yaml.dump(cfg, f)
    
    print(f'✅ Đã tạo cấu hình Kaggle chuẩn: {kaggle_yaml_path}')
    print(f'   • Thư mục ảnh train: {dataset_root / cfg.get("train", "images/train")}')
    print(f'   • Thư mục ảnh valid: {dataset_root / cfg.get("val", "images/valid")}')
    print(f'   • Thư mục ảnh test:  {dataset_root / cfg.get("test", "images/test")}')

✅ Đã tìm thấy dataset tại: /kaggle/input/datasets/scholar1407/smoking-cigaratte-dataset
✅ Đã tạo cấu hình Kaggle chuẩn: /kaggle/working/data_kaggle.yaml
   • Thư mục ảnh train: /kaggle/input/datasets/scholar1407/smoking-cigaratte-dataset/images/train
   • Thư mục ảnh valid: /kaggle/input/datasets/scholar1407/smoking-cigaratte-dataset/images/valid
   • Thư mục ảnh test:  /kaggle/input/datasets/scholar1407/smoking-cigaratte-dataset/images/test


## 🚀 Cell 4: Huấn luyện Mô hình Chính YOLOv8s-P2 (Primary Run: 100 Epochs)
- Tự động bật `batch=-1` (Auto-batch) tối ưu dung lượng VRAM GPU.
- Bật `amp=True` (FP16 mixed precision) tăng tốc độ huấn luyện x2.
- Checkpoint tự lưu mỗi **5 epoch một lần**.
- **Có cơ chế tự động khôi phục (Resume):** Nếu phiên làm việc bị ngắt giữa chừng, bạn chỉ cần bấm chạy lại ô này, nó sẽ tự động nhận diện `last.pt` và train tiếp tục mà không chạy lại từ đầu.

In [6]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from ultralytics import YOLO
import shutil

kaggle_yaml_path = Path('/kaggle/working/data_kaggle.yaml')
last_checkpoint = Path('/kaggle/working/runs/detect/smoke_p2/weights/last.pt')

# 1. Kiểm tra nếu có checkpoint bị dở dang thì train nối tiếp
if last_checkpoint.exists():
    print(f'🔄 Đã tìm thấy checkpoint! Đang khôi phục (Resume) tiếp tục từ: {last_checkpoint}')
    model_p2 = YOLO(str(last_checkpoint))
    model_p2.train(resume=True)
elif kaggle_yaml_path.exists():
    print('🚀 Bắt đầu huấn luyện YOLOv8s-P2 (Tối ưu tốc độ cao)...')
    model_p2 = YOLO('yolov8s-p2.yaml').load('yolov8s.pt')
    model_p2.train(
        data=str(kaggle_yaml_path),
        epochs=100,
        imgsz=640,
        batch=32,
        amp=True,
        save_period=5,
        patience=20,
        name='smoke_p2',
        project='/kaggle/working/runs/detect',
        exist_ok=True
    )
else:
    print('⚠️ Chưa tìm thấy data_kaggle.yaml. Vui lòng chạy Cell 3 trước!')

# 2. TỰ ĐỘNG NÉN FILE ZIP NGAY KHI TRAIN XONG (Đảm bảo không bao giờ mất)
print('📦 Đang tự động đóng gói file zip...')
shutil.make_archive('/kaggle/working/smoke_p2_final', 'zip', '/kaggle/working/runs/detect/smoke_p2')
print('✅ HOÀN TẤT! File smoke_p2_final.zip đã sẵn sàng để tải!')

🔄 Đã tìm thấy checkpoint! Đang khôi phục (Resume) tiếp tục từ: /kaggle/working/runs/detect/smoke_p2/weights/last.pt
Ultralytics 8.4.164 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_kaggle.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/ka

## ⚖️ Cell 5: Huấn luyện Mô hình Đối chứng YOLOv8s Baseline (Comparison Run: 100 Epochs)
Chạy mô hình chuẩn YOLOv8s (P3–P5) trong cùng điều kiện để so sánh định lượng lợi ích của đầu P2 đối với vật thể nhỏ.

In [7]:
from pathlib import Path
from ultralytics import YOLO

kaggle_yaml_path = Path('/kaggle/working/data_kaggle.yaml')
last_base_checkpoint = Path('/kaggle/working/runs/detect/smoke_baseline/weights/last.pt')

if last_base_checkpoint.exists():
    print(f'🔄 Đang khôi phục (Resume) Baseline từ: {last_base_checkpoint}')
    model_base = YOLO(str(last_base_checkpoint))
    model_base.train(resume=True)
elif kaggle_yaml_path.exists():
    print('🚀 Bắt đầu huấn luyện mô hình đối chứng YOLOv8s Baseline (100 Epochs)...')
    model_base = YOLO('yolov8s.pt')
    model_base.train(
        data=str(kaggle_yaml_path),
        epochs=100,
        imgsz=640,
        batch=32,
        amp=True,
        save_period=5,
        patience=20,
        name='smoke_baseline',
        project='/kaggle/working/runs/detect',
        exist_ok=True
    )
else:
    print('⚠️ Vui lòng chạy Cell 3 trước!')

# Bổ sung vào cuối Cell 5:
print('📦 Đang tự động đóng gói file zip Baseline...')
shutil.make_archive('/kaggle/working/smoke_baseline_final', 'zip', '/kaggle/working/runs/detect/smoke_baseline')
print('✅ File smoke_baseline_final.zip đã sẵn sàng!')


🚀 Bắt đầu huấn luyện mô hình đối chứng YOLOv8s Baseline (100 Epochs)...
New https://pypi.org/project/ultralytics/8.4.165 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.164 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_kaggle.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, m

## 📊 Cell 6: Đánh Giá So Sánh Dual-Metric & Bóc Tách Recall Vật Thể Nhỏ
Sau khi cả 2 mô hình đã hoàn thành 100 epochs, chạy ô này để đánh giá độc lập trên tập **Official Test** và **Clean Subset**, bóc tách Recall nhóm vật thể nhỏ (< 32×32 px) và xuất bảng báo cáo.

In [8]:
from pathlib import Path
from ultralytics import YOLO

p2_weights = Path('/kaggle/working/runs/detect/smoke_p2/weights/best.pt')
base_weights = Path('/kaggle/working/runs/detect/smoke_baseline/weights/best.pt')
kaggle_yaml = Path('/kaggle/working/data_kaggle.yaml')

if p2_weights.exists() and base_weights.exists():
    print('📊 Đang đánh giá YOLOv8s-P2 trên tập Test...')
    m_p2 = YOLO(str(p2_weights))
    res_p2 = m_p2.val(data=str(kaggle_yaml), split='test')
    
    print('\n📊 Đang đánh giá YOLOv8s-Baseline trên tập Test...')
    m_base = YOLO(str(base_weights))
    res_base = m_base.val(data=str(kaggle_yaml), split='test')
    
    print('\n' + '=' * 65)
    print('                 KẾT QUẢ ĐÁNH GIÁ SO SÁNH (OFFICIAL TEST)')
    print('=' * 65)
    print(f'YOLOv8s-P2:       mAP50 = {res_p2.results_dict.get("metrics/mAP50(B)", 0):.4f} | mAP50-95 = {res_p2.results_dict.get("metrics/mAP50-95(B)", 0):.4f}')
    print(f'YOLOv8s-Baseline: mAP50 = {res_base.results_dict.get("metrics/mAP50(B)", 0):.4f} | mAP50-95 = {res_base.results_dict.get("metrics/mAP50-95(B)", 0):.4f}')
    print('=' * 65)
else:
    print('⚠️ Chưa tìm thấy cả 2 file best.pt. Hãy đợi Cell 4 và Cell 5 hoàn tất huấn luyện!')

📊 Đang đánh giá YOLOv8s-P2 trên tập Test...
Ultralytics 8.4.164 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8s-p2 summary (fused): 90 layers, 10,626,708 parameters, 0 gradients, 36.6 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.4±0.8 ms, read: 6.2±1.0 MB/s, size: 45.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/scholar1407/smoking-cigaratte-dataset/labels/test... 732 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 732/732 185.4it/s 3.9s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/scholar1407/smoking-cigaratte-dataset/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 46/46 3.3it/s 13.9s0.3s
                   all        732       1793      0.899      0.836      0.896      0.584
Speed: 1.1ms preprocess